In [18]:
from datasets import load_dataset

ds = load_dataset("gmanolache/CrypticBio", split="train")

Resolving data files:   0%|          | 0/631 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/180 [00:00<?, ?it/s]

In [19]:
print(ds)
ds[0]

Dataset({
    features: ['scientificName', 'year', 'month', 'day', 'decimalLatitude', 'decimalLongitude', 'url', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'vernacularName', 'crypticGroup'],
    num_rows: 171832985
})


{'scientificName': 'Salticus scenicus',
 'year': 2022.0,
 'month': 3.0,
 'day': 16.0,
 'decimalLatitude': 49.682217,
 'decimalLongitude': -124.975088,
 'url': 'https://inaturalist-open-data.s3.amazonaws.com/photos/183187792/original.jpg',
 'kingdom': 'Animalia',
 'phylum': 'Arthropoda',
 'class': 'Arachnida',
 'order': 'Araneae',
 'family': 'Salticidae',
 'genus': 'Salticus',
 'vernacularName': ['Zebra Jumper',
  'Zebra Spider',
  'Zebra Jumping Spider',
  'Zebra Back Spider'],
 'crypticGroup': ['Phidippus audax',
  'Salticus cingulatus',
  'Paraphidippus aurantius',
  'Platycryptus undatus',
  'Attulus fasciger',
  'Naphrys pulex',
  'Marpissa muscosa',
  'Eris militaris',
  'Menemerus semilimbatus',
  'Menemerus bivittatus',
  'Hasarius adansoni',
  'Phidippus putnami',
  'Pelegrina galathea',
  'Maevia inclemens',
  'Pseudeuophrys lanigera',
  'Phidippus otiosus']}

# We need to filter to a smaller sub-set of data so that experimetns are more managable 

In [20]:

#Load just the benchmark files
common =load_dataset("gmanolache/CrypticBio", data_files="CrypticBio-Benchmark/CrypticBio-Common.csv", split="train")
print(common.features)

{'scientificName': Value('string'), 'year': Value('float64'), 'month': Value('float64'), 'day': Value('float64'), 'decimalLatitude': Value('float64'), 'decimalLongitude': Value('float64'), 'url': Value('string'), 'kingdom': Value('string'), 'phylum': Value('string'), 'class': Value('string'), 'order': Value('string'), 'family': Value('string'), 'genus': Value('string'), 'vernacularName': Value('string')}


In [21]:
#get unique pairs from the benchmark
unique_names = set(common["scientificName"])

#batch filtering to get the common subset
common_subset = ds.filter(
    lambda x: [
        name in unique_names 
        for name in x['scientificName']
    ],
    batched=True,
    batch_size=10000
)


In [22]:
import requests
from PIL import Image
from io import BytesIO
import torch
import numpy as np


# Open the image from the URL
def open_image(url : list[str]):
    try:
        response = requests.get(url, timeout=10)
        return Image.open(BytesIO(response.content))
    except requests.exceptions.RequestException as e:
        print(f"Error downloading image: {e}")
        return None

# Example usage
img_url = common[12985]['url']
image = open_image(img_url)
image.show()

In [23]:
import open_clip

# Load the model and preprocessors
model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms('hf-hub:imageomics/bioclip-2')
tokenizer = open_clip.get_tokenizer('hf-hub:imageomics/bioclip-2')


#tojenize the text from family and genus - same as cryptic group names
def tokenize_family_genius(sample: dict):
    family_name = sample['family']
    genus_name = sample['genus']
    return tokenizer(family_name + ' ' + genus_name)

#preprocess the image and add batch dimension  
def preprocess_image(sample: dict):
    img_url = sample['url']
    image = open_image(img_url)
    if image is not None:
        return preprocess_val(image).unsqueeze(0)  #add batch dimension
    else:
        return None






In [24]:
#take a random list of uniqeue familises 
list_of_uniqe_families = set(common_subset['scientificName'][np.random.choice(len(common_subset), size=200, replace=False)]) 
list_of_uniqe_families = list(list_of_uniqe_families)[:9] 


# Test model on species recognition randomly sampled


In [25]:


sample_index = np.random.randint(0, len(common_subset))
example = common_subset[sample_index]
list_of_inptut_species = list(set(list_of_uniqe_families + [example['scientificName']]))

gt_index = list_of_inptut_species.index(example['scientificName'])

tokenized_text = tokenizer(list_of_inptut_species)
image_features = preprocess_image(example)

prompt_template = "A photo of a "  # same as cryptic group names
prompts = [f"{prompt_template}{name}" for name in list_of_inptut_species]
tokenized_text = tokenizer(prompts)

image_features = model.encode_image(image_features)
text_features = model.encode_text(tokenized_text)
image_features /= image_features.norm(dim=-1, keepdim=True)
text_features /= text_features.norm(dim=-1, keepdim=True)
text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)

print(f"Ground truth index: {gt_index}")
print("Label probs:", text_probs) 
print(f"Ground truth: {example['scientificName']}")
print(f"Ground truth prob: {text_probs[0, gt_index].item():.4f}")
print(f"Predicted: {list_of_inptut_species[text_probs.argmax().item()]}")
print(f"Correct: {text_probs.argmax().item() == gt_index}")

Ground truth index: 9
Label probs: tensor([[2.5789e-10, 3.6360e-10, 1.2541e-08, 8.8049e-08, 4.1345e-07, 1.4779e-06,
         4.1325e-07, 1.2490e-05, 9.1064e-09, 9.9999e-01]],
       grad_fn=<SoftmaxBackward0>)
Ground truth: Fringilla coelebs
Ground truth prob: 1.0000
Predicted: Fringilla coelebs
Correct: True


# Test model on species that are confusing


In [26]:
#move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)



sample_index = np.random.randint(0, len(common_subset))
example = common_subset[sample_index]
list_of_confusing_species = list(example['crypticGroup'])
list_of_inptut_species = list(set(list_of_confusing_species + [example['scientificName']]))

gt_index = list_of_inptut_species.index(example['scientificName'])

tokenized_text = tokenizer(list_of_inptut_species)
image_features = preprocess_image(example)

prompt_template = "A photo of a "  # same as cryptic group names
prompts = [f"{prompt_template}{name}" for name in list_of_inptut_species]
tokenized_text = tokenizer(prompts).to(device)
image_features = image_features.to(device)  
with torch.no_grad(), torch.amp.autocast("cuda"):
    image_features = model.encode_image(image_features)
    text_features = model.encode_text(tokenized_text)
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)
    text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)

print(f"Ground truth index: {gt_index}")
print("Label probs:", text_probs) 
print(f"Ground truth: {example['scientificName']}")
print(f"Ground truth prob: {text_probs[0, gt_index].item():.4f}")
print(f"Predicted: {list_of_inptut_species[text_probs.argmax().item()]}")
print(f"Correct: {text_probs.argmax().item() == gt_index}")

Ground truth index: 8
Label probs: tensor([[1.7008e-05, 3.0148e-04, 4.6694e-04, 2.2757e-04, 3.8329e-05, 2.5494e-02,
         1.4526e-02, 3.3825e-05, 9.5665e-01, 3.1775e-05, 2.9556e-06, 1.2567e-04,
         7.2321e-04, 7.8645e-05, 2.7179e-05, 2.2532e-05, 8.3717e-05, 6.2570e-06,
         3.7519e-04, 3.3825e-05, 1.2966e-04, 1.6649e-04, 8.0341e-06, 2.2057e-04,
         2.0720e-04]], device='cuda:0')
Ground truth: Melospiza melodia
Ground truth prob: 0.9567
Predicted: Melospiza melodia
Correct: True


## Experiment 1 - adding Date (month + day) to the existent embeddings 

Encoding the date(month,day), using sine and cosine following the intuition from: https://harrisonpim.com/blog/the-best-way-to-encode-dates-times-and-other-cyclical-features . We kinda mimic the first stage of the BioClip training - learing meaningful represemntation of date

In [27]:
def encode_date(month, day):
    """
    Encode month and day as cyclical features using sin/cos projections.
    Each cyclical component maps to a point on a unit circle,
    so December (12) is close to January (1), and day 365 is close to day 1.
    """
    # Normalize to [0, 1] range
    month_norm = (month - 1) / 12       
    day_of_year_norm = ((month - 1) * 30 + day) / 365 

    # Project onto unit circle using sin/cos
    month_sin = (np.sin(2 * np.pi * month_norm) + 1) / 2
    month_cos = (np.cos(2 * np.pi * month_norm) + 1) / 2
    day_sin = (np.sin(2 * np.pi * day_of_year_norm) + 1) / 2
    day_cos = (np.cos(2 * np.pi * day_of_year_norm) + 1) / 2

    return torch.tensor([month_sin, month_cos, day_sin, day_cos], dtype=torch.float32)

In [28]:
example = common_subset[0]
date_features = encode_date(example['month'], example['day'])
print(date_features)

tensor([0.5000, 0.0000, 0.3682, 0.0177])


Having designed our embeding space, we try to map the date to the same embeddiong space as the CLIP representations for image and text.

In [34]:
import torch.nn as nn

# This network is prett naive and prob can be improved

class DateEmbedder(nn.Module):
    """Maps 4-dim date features into BioCLIP's 512-dim space."""
    def __init__(self, date_dim=4, clip_dim=768):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(date_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, clip_dim),
        )
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

    def forward(self, x):
        emb = self.network(x)
        return emb / emb.norm(dim=-1, keepdim=True)  # L2 normalize to match CLIP space

**The idea for now is the following: Using the learned text represetnations from taxa, we can try to align the date as with it. Thus due to the rule of assosiativity, we can avoid the slow process of training with images! - although its probably a good idea to redsign the experiment**

In [30]:
from tqdm import tqdm #to monitor progress


scientific_names = common_subset['scientificName']

# Order doesn't matter here because you dedupe next
unique_names = sorted(set(scientific_names))
name_to_idx = {name: i for i, name in enumerate(unique_names)}


# Encode using BioCLI in batches
BATCH = 256
species_text_embs = []
with torch.no_grad(): 
    for i in tqdm(range(0, len(unique_names), BATCH), desc="Encoding names"):
        batch = unique_names[i:i+BATCH]
        tokens = tokenizer(batch).to(device)
        #normalize the embeddings to unit length to match CLIP space
        embs = model.encode_text(tokens)
        embs /= embs.norm(dim=-1, keepdim=True)
        species_text_embs.append(embs.cpu()) #move to CPU to save GPU memory

# Concatenate all batches into a single tensor
species_text_embs = torch.cat(species_text_embs, dim=0)


Encoding names: 100%|██████████| 1/1 [00:00<00:00,  2.05it/s]


In [31]:
print(len(unique_names))
print(len(scientific_names))


158
6497834


In [32]:
# from tqdm import tqdm #to monitor progress

# # returns the full name of the taxonomy for a sample
# def taxonomy_string(sample):
#     return f"{sample['kingdom']} {sample['phylum']} {sample['class']} {sample['order']} {sample['family']} {sample['genus']}"

# # Collect unique taxonomy strings per species
# species_to_taxonomy = {}
# for sample in tqdm(common_subset, desc="Collecting taxonomy"):
#     name = sample['scientificName']
#     if name not in species_to_taxonomy:
#         species_to_taxonomy[name] = taxonomy_string(sample)

# # Sort species and corresponding taxonomy strings for consistent ordering
# all_species = sorted(species_to_taxonomy.keys())
# # Create a list of taxonomy strings in the same order as species
# all_taxonomy_strings = [species_to_taxonomy[s] for s in all_species]

# # Encode using BioCLI in batches
# BATCH = 256
# species_text_embs = []
# with torch.no_grad(): 
#     for i in tqdm(range(0, len(all_taxonomy_strings), BATCH), desc="Encoding taxonomy"):
#         batch = all_taxonomy_strings[i:i+BATCH]
#         tokens = tokenizer(batch).to(DEVICE)
#         #normalize the embeddings to unit length to match CLIP space
#         embs = model.encode_text(tokens)
#         embs /= embs.norm(dim=-1, keepdim=True)
#         species_text_embs.append(embs.cpu()) #move to CPU to save GPU memory

# # Concatenate all batches into a single tensor
# species_text_embs = torch.cat(species_text_embs, dim=0)
# # Create a mapping from species name to index in the embedding tensor
# label_to_idx = {s: i for i, s in enumerate(all_species)}
# print(f"{len(all_species)} species, shape: {species_text_embs.shape}")

In [ ]:
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
import torch.nn.functional as F
samples_indecies=np.random.choice(len(common_subset), size=1_000_000, replace=False)
common_subset_sub = common_subset.select(samples_indecies)
# Precompute all date tensors and labels into memory for fast access during training
all_dates = [] 
all_labels = [] 
months = common_subset_sub['month']
days   = common_subset_sub['day']
names  = common_subset_sub['scientificName']

all_dates, all_labels = [], []
for m, d, n in tqdm(zip(months, days, names), total=len(months), desc="Precomputing"):
    if m is None or d is None:
        continue
    all_dates.append(encode_date(m, d))
    all_labels.append(name_to_idx[n])
#mix the dates so that we have a nice split of the data, and to avoid any potential ordering bias during training
train_idx, test_idx = train_test_split(
    range(len(all_dates)),
    test_size=0.2,
    stratify=np.asarray(all_labels),
    random_state=42
)
#Create train and test splits
all_dates  = torch.stack([torch.as_tensor(d, dtype=torch.float32) for d in all_dates])
all_labels = torch.as_tensor(all_labels, dtype=torch.long)

train_dates,  test_dates  = all_dates[train_idx],  all_dates[test_idx]
train_labels, test_labels = all_labels[train_idx], all_labels[test_idx]

# define a dataloader for the training set, maybe batch size is too big, I increased it to speed up training
train_loader = DataLoader(
    torch.utils.data.TensorDataset(train_dates, train_labels),
    batch_size=2048, shuffle=True
)

# Train  - TODO find a way to make run faster
date_embedder = DateEmbedder().to(device)
optimizer = torch.optim.Adam(date_embedder.parameters(), lr=1e-3)
species_text_embs_gpu = species_text_embs.to(device)

for epoch in range(30):
    total_loss = 0
    steps = 0


    for date_inputs, label_indices in train_loader:
        date_inputs   = date_inputs.to(device)
        label_indices = label_indices.to(device)

        target_text = species_text_embs_gpu[label_indices]        # (B, D), unit-norm
        date_embs   = date_embedder(date_inputs)                  # (B, D)

        #contrastive loss: maximize cosine similarity between date_embs and target_text
        date_embs_norm = F.normalize(date_embs, dim=-1)
        target_text_norm = F.normalize(target_text, dim=-1)
       # Create the n x n similarity matrix
        logit_scale = date_embedder.logit_scale.exp()
        logits = (date_embs_norm @ target_text_norm.T) * torch.exp(logit_scale)

        # Labels are just the diagonal (0, 1, 2, ..., n-1)
        labels = torch.arange(n)

        # Symmetric loss
        loss_i = F.cross_entropy(logits, labels)
        loss_t = F.cross_entropy(logits.T, labels)
        loss = (loss_i + loss_t) / 2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        steps += 1
    
    print(f"Epoch {epoch+1} | Loss: {total_loss/steps:.4f}")

Precomputing:  49%|████▉     | 487946/1000000 [00:46<00:48, 10482.26it/s]


KeyboardInterrupt: 

**The idea is now that if I take image metadata (such as date), I could better defirentiate the species**

In [17]:
def predict(image_emb, date_emb, species_text_embs, date_encoder, alpha=0.8):
    """
    image_emb:         (1, 512) from BioCLIP
    date_emb:          (1, 4) cyclical features
    species_text_embs: (N, 512) precomputed for all candidate species
    alpha:             weight for image vs date (tune this)
    """
    with torch.no_grad():
        date_in_clip = date_encoder(date_emb)  # (1, 512)

        image_sim = image_emb @ species_text_embs.T  # (1, N)
        date_sim  = date_in_clip @ species_text_embs.T  # (1, N)

        combined = alpha * image_sim + (1 - alpha) * date_sim
        probs = combined.softmax(dim=-1)

    return probs

# Experiment 2? Same Idea but with location of the image! 